In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy
from collections import namedtuple

device='cuda' if torch.cuda.is_available() else 'cpu'

print(f"using device: {device}")
if device=='cuda':
  gpu_name=torch.cuda.get_device_name(0)

  total_memory=torch.cuda.get_device_properties(0).total_memory

  total_memory_gb=total_memory/(1024**3)

  print("GPU", gpu_name)
  print("Total memory VRAM: ", total_memory_gb)


using device: cuda
GPU Tesla T4
Total memory VRAM:  14.56317138671875


In [18]:
class ReLUConvBN(nn.Module):
  def __init__(self, C_in, C_out, kernel_size, stride, padding, affine):
    super().__init__()
    self.relu=nn.ReLU(inplace=False)
    self.conv=nn.Conv2d(C_in, C_out, kernel_size, stride, padding, bias=False)
    self.bn=nn.BatchNorm2d(C_out, affine=affine)
  def forward(self, x):
    x=self.relu(x)
    x=self.conv(x)
    x=self.bn(x)
    return x



class SepConv(nn.Module):
  def __init__(self, C_in, C_out, kernel_size, stride, padding, affine):
    super().__init__()
    self.relu=nn.ReLU(inplace=False)
    self.conv_depthwise=nn.Conv2d(C_in, C_in, kernel_size, stride, padding, groups=C_in, bias=False)
    self.conv_pointwise=nn.Conv2d(C_in, C_out, kernel_size=1, bias=False)
    self.bn=nn.BatchNorm2d(C_out, affine=affine)
    self.relu2=nn.ReLU(inplace=False)
    self.conv2_depthwise=nn.Conv2d(C_out, C_out, kernel_size, stride=1, padding=padding, groups=C_out, bias=False)
    self.conv2_pointwise=nn.Conv2d(C_out, C_out, kernel_size=1, bias=False)
    self.bn2=nn.BatchNorm2d(C_out, affine=affine)
  def forward(self, x):
    x=self.relu(x)
    x=self.conv_depthwise(x)
    x=self.conv_pointwise(x)
    x=self.bn(x)
    x=self.relu2(x)
    x=self.conv2_depthwise(x)
    x=self.conv2_pointwise(x)
    x=self.bn2(x)
    return x



class DilConv(nn.Module):
  def __init__(self, C_in, C_out, kernel_size, stride, padding, dilation, affine):
    super().__init__()
    self.relu=nn.ReLU(inplace=False)
    self.conv_depthwise=nn.Conv2d(C_in, C_in, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=C_in, bias=False)
    self.conv_pointwise=nn.Conv2d(C_in, C_out, kernel_size=1, bias=False)
    self.bn=nn.BatchNorm2d(C_out, affine=affine)
  def forward(self, x):
    x=self.relu(x)
    x=self.conv_depthwise(x)
    x=self.conv_pointwise(x)
    x=self.bn(x)
    return x

class Identity(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self, x):
    return x

class Zero(nn.Module):
  def __init__(self, stride):
    super().__init__()
    self.stride=stride
  def forward(self, x):
    x=x[:, :, ::self.stride, ::self.stride]
    x=x*0
    return x

class FactorizedReduce(nn.Module):
  def __init__(self, C_in, C_out, affine=True):
    super().__init__()
    self.relu=nn.ReLU(inplace=False)
    self.conv_1_by_1=nn.Conv2d(C_in, C_out//2, kernel_size=1, stride=2,  bias=False)
    self.conv_1_by_1_2=nn.Conv2d(C_in, C_out//2, kernel_size=1, stride=2,  bias=False)
    self.bn=nn.BatchNorm2d(C_out, affine=affine)
  def forward(self, x):
    x=self.relu(x)
    x_new_1=self.conv_1_by_1(x)
    x_new_2=self.conv_1_by_1_2(x[:, :, 1:, 1:])
    x_resultant=torch.cat([x_new_1, x_new_2], 1)
    x_resultant=self.bn(x_resultant)
    return x_resultant

In [19]:
OPS={
    'none': lambda C, stride, affine: Zero(stride),
    'avg_pool_3x3': lambda C, stride, affine: nn.Sequential(nn.AvgPool2d(3, stride=stride, padding=1, count_include_pad=False), nn.BatchNorm2d(C, affine=affine)),
    'max_pool_3x3': lambda C, stride, affine: nn.Sequential(nn.MaxPool2d(3, stride=stride, padding=1), nn.BatchNorm2d(C, affine=affine)),
    'skip_connect': lambda C, stride, affine: Identity() if stride==1 else FactorizedReduce(C, C, affine=affine),
    'sep_conv_3x3': lambda C, stride, affine: SepConv(C, C, 3, stride, 1, affine),
    'sep_conv_5x5': lambda C, stride, affine: SepConv(C, C, 5, stride, 2, affine),
    'dil_conv_3x3': lambda C, stride, affine: DilConv(C, C, 3, stride, 2, 2, affine),
    'dil_conv_5x5': lambda C, stride, affine: DilConv(C, C, 5, stride, 4, 2, affine),
}

In [20]:
x=torch.randn(2, 16, 32, 32)
for key in OPS.keys():
  op = OPS[key](C=16, stride=1, affine=True)
  x_resultant=op(x)
  print(key, x_resultant.shape)
for key in OPS.keys():
  op=OPS[key](C=16, stride=2, affine=True)
  x_resultant=op(x)
  print(key, x_resultant.shape)

none torch.Size([2, 16, 32, 32])
avg_pool_3x3 torch.Size([2, 16, 32, 32])
max_pool_3x3 torch.Size([2, 16, 32, 32])
skip_connect torch.Size([2, 16, 32, 32])
sep_conv_3x3 torch.Size([2, 16, 32, 32])
sep_conv_5x5 torch.Size([2, 16, 32, 32])
dil_conv_3x3 torch.Size([2, 16, 32, 32])
dil_conv_5x5 torch.Size([2, 16, 32, 32])
none torch.Size([2, 16, 16, 16])
avg_pool_3x3 torch.Size([2, 16, 16, 16])
max_pool_3x3 torch.Size([2, 16, 16, 16])
skip_connect torch.Size([2, 16, 16, 16])
sep_conv_3x3 torch.Size([2, 16, 16, 16])
sep_conv_5x5 torch.Size([2, 16, 16, 16])
dil_conv_3x3 torch.Size([2, 16, 16, 16])
dil_conv_5x5 torch.Size([2, 16, 16, 16])


In [21]:
class MixedOp(nn.Module):
  def __init__(self, C, stride):
    super().__init__()
    self.ops=nn.ModuleList()
    for primitive in OPS.keys():
      op=OPS[primitive](C, stride, affine=False)
      self.ops.append(op)
  def forward(self, x, weights):
    weighted_sum=sum(w*op(x) for w, op in zip(weights, self.ops))
    return weighted_sum

In [22]:
instance=MixedOp(C=16, stride=1)
random_weights = torch.randn(8, requires_grad=True)
softmax_weights = F.softmax(random_weights, dim=0)
forward_pass = instance(x, softmax_weights)
forward_pass.sum().backward()
print(softmax_weights)
print(forward_pass.shape)
print(random_weights.grad)

tensor([0.1227, 0.1959, 0.0820, 0.0530, 0.1661, 0.3009, 0.0394, 0.0400],
       grad_fn=<SoftmaxBackward0>)
torch.Size([2, 16, 32, 32])
tensor([-0.9603, -1.5328, -0.6420,  7.4108, -1.2995, -2.3549, -0.3082, -0.3132])


In [23]:
class Cell(nn.Module):
  def __init__(self,  steps, C_prev, C_prev_prev, C, reduction, reduction_prev):
    super().__init__()
    #self.s0=s0
    #self.s1=s1
    self.reduction=reduction
    if reduction_prev:
      self.preprocess_conv2d=FactorizedReduce(C_prev_prev, C, affine=False)
    else:
      self.preprocess_conv2d=ReLUConvBN(C_prev_prev, C, 1, 1, 0, affine=False)
    self.preprocess_conv2d_2=ReLUConvBN(C_prev, C, 1, 1, 0, affine=False)
    self.steps=steps
    self.edges=nn.ModuleList()
    #2 + 3 + 4 + 5 -> 14
    for i in range(steps):
      for j in range(i+2):
        stride=2 if reduction and j<2 else 1
        op=MixedOp(C, stride)
        self.edges.append(op)
    #each intermediate node receives input from previous one via MixedOp

    #reduction cells use stride=2 on edges connected to inputs s1,s0 everywhere else it is stride=1

    #output is torch.cat of all previous 4 intermediate nodes along channel dim

  def forward(self, x_s0, x_s1, weights):
    x_s0=self.preprocess_conv2d(x_s0)
    x_s1=self.preprocess_conv2d_2(x_s1)
    states=[x_s0, x_s1]
    offset=0
    for i in range(self.steps):
      s=sum(
          self.edges[offset + j](states[j], weights[offset + j])
            for j in range(len(states))
      )
      offset+=len(states)
      states.append(s)
    #node 2 -> node 3 -> node 4 -> node 5
    return torch.cat(states[2:], dim=1)


In [24]:
s0 = torch.randn(2, 48, 32, 32)
s1 = torch.randn(2, 48, 32, 32)

weights = F.softmax(torch.randn(14, 8), dim=-1)

cell_normal=Cell( 4, 48, 48, 16, False, False)
cell_reduce=Cell( 4,  48, 48, 16, True, False)

cell_normal_out=cell_normal(s0, s1, weights)
cell_reduce_out=cell_reduce(s0, s1, weights)

print(cell_normal_out.shape)
print(cell_reduce_out.shape)

torch.Size([2, 64, 32, 32])
torch.Size([2, 64, 16, 16])


In [25]:
class Search_Network(nn.Module):
  def __init__(self, C_init, num_classes,  total_cells=8, steps=4):
    super().__init__()
    C_current=C_init
    C_prev=C_init*3
    C_prev_prev=C_init*3
    reduction_prev=False
    self.stem=nn.Sequential(nn.Conv2d(3, C_init*3, 3, padding=1, bias=False), nn.BatchNorm2d(C_init*3))
    self.cells=nn.ModuleList()
    for i in range(total_cells):
      reduction= (i==total_cells//3 or i==2*total_cells//3)
      cell=Cell(steps, C_prev, C_prev_prev, C_current, reduction, reduction_prev)
      self.cells.append(cell)
      C_prev_prev=C_prev
      C_prev=C_current*4
      if reduction:
        C_current*=2
      reduction_prev=reduction

    self.alpha_normal=nn.Parameter(torch.randn(14,8))
    self.alpha_reduce=nn.Parameter(torch.randn(14,8))
    self.global_pooling=nn.AdaptiveAvgPool2d(1)
    self.classifier=nn.Linear(C_prev, num_classes)
  def arch_parameters(self):
    return self.alpha_normal, self.alpha_reduce
  def forward(self, x):
    weights_normal=F.softmax(self.alpha_normal, dim=-1)
    weights_reduce=F.softmax(self.alpha_reduce, dim=-1)
    s1=s0=self.stem(x)
    for cell in self.cells:
      if cell.reduction:
        s0, s1=s1, cell(s0, s1, weights_reduce)
      else:
        s0, s1=s1, cell(s0, s1, weights_normal)
    s1=self.global_pooling(s1)
    s1=s1.view(s1.size(0), -1)
    logits=self.classifier(s1)
    return logits

In [26]:
model = Search_Network(C_init=16, num_classes=10)
x = torch.randn(4, 3, 32, 32)
logits = model(x)
print(logits.shape)

# count parameters
total_params = sum(p.numel() for p in model.parameters())
arch_params = sum(p.numel() for p in model.arch_parameters())
weight_params = total_params - arch_params

print(f"Total params: {total_params}")
print(f"Arch params: {arch_params}")
print(f"Weight params: {weight_params}")

torch.Size([4, 10])
Total params: 1478298
Arch params: 224
Weight params: 1478074


In [29]:
class Architect_first_order:
  def __init__(self, model, arch_lr, arch_weight_decay):
    super().__init__()
    self.model=model
    self.optimizer=torch.optim.Adam(self.model.arch_parameters(), lr=arch_lr, weight_decay=arch_weight_decay)
  def step(self, x_val, y_val):
    self.optimizer.zero_grad()
    logits=self.model(x_val)
    loss=F.cross_entropy(logits, y_val)
    loss.backward()
    self.optimizer.step()


In [31]:
model = Search_Network(C_init=16, num_classes=10)
architect = Architect_first_order(model, arch_lr=3e-4, arch_weight_decay=1e-3)

x_val = torch.randn(4, 3, 32, 32)
y_val = torch.randint(0, 10, (4,))

alpha_before = model.alpha_normal.clone()
architect.step(x_val, y_val)
alpha_after = model.alpha_normal.clone()

print(torch.allclose(alpha_before, alpha_after))  # should print False

False


In [38]:
class Architect_second_order(Architect_first_order):
  def step(self, x_val, y_val, x_train, y_train, xi):
    self.optimizer.zero_grad()
    arch_params = set(self.model.arch_parameters())
    w = [p for p in self.model.parameters() if p not in arch_params]

    logits_train=self.model(x_train)
    train_loss=F.cross_entropy(logits_train, y_train)
    #w=list(self.model.parameters())
    dw=torch.autograd.grad(train_loss, w)
    w_prime=[p - xi*d for p, d in zip(w, dw)]

    original_weights=[p.data.clone() for p in self.model.parameters()]

    for p, p_prime in zip(w, w_prime):
      p.data=p.data.copy_(p_prime)

    logits_val=self.model(x_val)
    val_loss=F.cross_entropy(logits_val, y_val)



    d_w_prime = torch.autograd.grad(val_loss, w, retain_graph=True)
    d_alpha=torch.autograd.grad(val_loss, self.model.arch_parameters())

    for p, p_original in zip(self.model.parameters(), original_weights):
      p.data.copy_(p_original)

    epsilon=0.01/torch.cat([g.flatten() for g in d_w_prime]).norm()

    w_plus  = [p + epsilon * g for p, g in zip(w, d_w_prime)]
    w_minus = [p - epsilon * g for p, g in zip(w, d_w_prime)]

    for p, p_w_plus in zip(w, w_plus):
      p.data.copy_(p_w_plus)

    logits_plus=self.model(x_train)
    plus_loss=F.cross_entropy(logits_plus, y_train)
    grad_alpha_plus=torch.autograd.grad(plus_loss, self.model.arch_parameters())

    for p, p_original in zip(self.model.parameters(), original_weights):
      p.data.copy_(p_original)

    for p, p_w_minus in zip(w, w_minus):
      p.data.copy_(p_w_minus)

    logits_minus=self.model(x_train)
    minus_loss=F.cross_entropy(logits_minus, y_train)
    grad_alpha_minus=torch.autograd.grad(minus_loss, self.model.arch_parameters())

    for p, p_original in zip(self.model.parameters(), original_weights):
      p.data.copy_(p_original)

    correction_term=[(g_plus-g_minus)/(2*epsilon) for g_plus, g_minus in zip(grad_alpha_plus, grad_alpha_minus)]

    final_grad=[dv - xi*c for dv, c in zip(d_alpha, correction_term)]

    for param, grad in zip(self.model.arch_parameters(), final_grad):
      param.grad=grad
    self.optimizer.step()

In [39]:
model = Search_Network(C_init=16, num_classes=10)
architect = Architect_second_order(model, arch_lr=3e-4, arch_weight_decay=1e-3)

x_train = torch.randn(4, 3, 32, 32)
y_train = torch.randint(0, 10, (4,))
x_val = torch.randn(4, 3, 32, 32)
y_val = torch.randint(0, 10, (4,))

alpha_before = model.alpha_normal.clone()
architect.step(x_val, y_val, x_train, y_train, xi=0.01)
alpha_after = model.alpha_normal.clone()

print(torch.allclose(alpha_before, alpha_after))  # should print False

False


In [40]:
import torchvision
import torchvision.transforms as transforms

CIFAR_MEAN = [0.49139968, 0.48215827, 0.44653124]
CIFAR_STD  = [0.24703233, 0.24348505, 0.26158768]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

from torch.utils.data import random_split

full_train = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)

n = len(full_train)  # 50000
train_data, val_data = random_split(full_train, [n//2, n//2])

val_data.dataset.transform = val_transform

train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
    num_workers=0
)

val_loader = torch.utils.data.DataLoader(
    val_data,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
    num_workers=0
)



100%|██████████| 170M/170M [00:05<00:00, 32.7MB/s]


In [ ]:
model=Search_Network(C_init=16, num_classes=10).to(device)
architect=Architect_second_order(model, arch_lr=3e-4, arch_weight_decay=1e-3)
weight_params = [p for p in model.parameters() if p not in set(model.arch_parameters())]
optimizer=torch.optim.SGD(weight_params, lr=0.025, momentum=0.9, weight_decay=3e-4)
cosine_scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,  T_max=50, eta_min=0)

for epoch in range(50):
  for (batch_val, label_val), (batch_train, label_train) in zip(val_loader, train_loader):
    batch_train = batch_train.to(device)
    label_train = label_train.to(device)
    batch_val   = batch_val.to(device)
    label_val   = label_val.to(device)
    train_result=model(batch_train)
    loss=F.cross_entropy(train_result, label_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    architect.step(batch_val, label_val, batch_train, label_train, xi=0.025)
  cosine_scheduler.step()
  print(f"Epoch {epoch} | Loss {loss.item():.4f}")